In [3]:
!pip install ollama 

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Cel 1: Imports en configuratie
import ollama
import json
from dotenv import load_dotenv

load_dotenv()

# Test of Ollama bereikbaar is
models = ollama.list()
print("Beschikbare modellen:")
for m in models['models']:
    print(f"  - {m['model']}")

Beschikbare modellen:
  - qwen2.5:7b


In [6]:
response = ollama.chat(
    model="qwen2.5:7b", 
    messages=[
        {"role":"user", "content":"Wat is TSH"}
    ]
)
print(response['message']['content'])

TSH stands for Thyroid-Stimulating Hormone. It is produced by the anterior pituitary gland and plays a crucial role in regulating the function of the thyroid gland. Here are some key points about TSH:

1. Function: TSH stimulates the thyroid gland to produce and release thyroxine (T4) and triiodothyronine (T3), which are important hormones that regulate metabolism.

2. Regulation: The production of TSH is controlled by a feedback mechanism involving the hypothalamus, pituitary gland, and thyroid gland.

3. Testing: A TSH blood test is often used to diagnose and monitor conditions related to thyroid function, such as hypothyroidism (underactive thyroid) or hyperthyroidism (overactive thyroid).

4. Normal range: The normal range for TSH can vary slightly depending on the laboratory, but it typically falls between 0.4 mIU/L and 4.0 mIU/L.

5. Clinical significance: Elevated levels of TSH may indicate hypothyroidism, while low levels can suggest hyperthyroidism or an overactive thyroid gla

In [8]:
SYSTEM_PROMPT = """ Je bent een klinisch chemicus die laboratoriumuitslagen interpreteert 
voor artsen. Je geeft altijd: 
1. Een korte interpretatie van de waarden
2. De meest waarschijnlijke diagnose 
3. Een aanbeveling voor vervolgbeleid 

Je antwoorden zijn bondig, klinisch, en evidence-based """ 

response = ollama.chat(
    model="qwen2.5:7b", 
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}, 
        {"role": "user", "content": "TSH: 8.2 mU/L (ref: 0.4-4.0), fT4: 9.1 pmol/L (ref: 10-24). Patiënt: vrouw, 45 jaar, klachten van vermoeidheid en kouwelijkheid."}
    ]
)
print(response["message"]["content"])

1. **Interpretatie**:
   - De TSH waarde is aanwijzingen van hypothyreïdie (TSH >4.0 mU/L), terwijl de fT4 ligt in het laagseinde van normaal.
   
2. **Meest waarschijnlijke diagnose**:
   - Subklare hypothyreïdie of lichte hypothyreïdie.

3. **Aanbeveling voor vervolgbeleid**:
   - Overweeg de fT3 te meten, aangezien TSH en fT4 alleen soms tegengesteld reageren.
   - Als fT3 ook laag is, kan substitutietherapie met thyroïdeerhormoon (thyroxine) worden overwogen.


In [9]:
# Cel 4: Few-shot prompting - voorbeelden meegeven
FEW_SHOT_PROMPT = """Je bent een klinisch chemicus. Interpreteer lab-uitslagen 
in het volgende vaste formaat:

VOORBEELD 1:
Input: TSH: 0.05 mU/L, fT4: 32 pmol/L, man 55 jaar
Output:
- Interpretatie: Suppressed TSH met verhoogd fT4 wijst op hyperthyreoïdie
- Waarschijnlijke diagnose: Hyperthyreoïdie (bijv. Graves, toxisch adenoom)
- Beleid: fT3 bepalen, TSH-receptor antilichamen, verwijzing endocrinologie

VOORBEELD 2:
Input: Natrium: 128 mmol/L, osmolaliteit 268 mOsm/kg, urine natrium 45 mmol/L
Output:
- Interpretatie: Hypotone hyponatriëmie met verhoogd urinenatrium, euvolemisch
- Waarschijnlijke diagnose: SIADH
- Beleid: Vochtbeperking, onderliggende oorzaak zoeken (maligniteit, medicatie, CNS)

Interpreteer nu de volgende waarden:"""

test_case = "Kalium: 2.9 mmol/L, Natrium: 141 mmol/L, Creatinine: 88 µmol/L, patiënt gebruikt diuretica"

response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {"role": "system", "content": FEW_SHOT_PROMPT},
        {"role": "user", "content": test_case}
    ]
)

print(response['message']['content'])

- Interpretatie: Loodzwaar kalieemi met normaal natrieemi en laag creatinieniveau suggereren hypokaliëmie. Hoewel de creatinienivool ligt in het normale interval, kan dit bij zekerheid een indirecte indicatie zijn van wateren wisseling of euvolemie.
- Waarschijnlijke diagnose: Hypokaliëmie
- Beleid: Evalueer symptomen zoals spierzwakte, irritabiliteit. Beperking van kaliumverlies door bijvoorbeeld diuretica. Controle met 24uurtotaaluriner Kalium. Potiële oorzaken zoeken (aanvaarbaar Kaliumvoorraad, loodzware etioologie) en behandel de oorzaak wanneer deze duidelijk is.


In [10]:
# Cel 5: Vergelijking - zie zelf het verschil
same_case = "Kalium: 2.9 mmol/L, Natrium: 141 mmol/L, Creatinine: 88 µmol/L, patiënt gebruikt diuretica"

# Zero-shot (geen voorbeelden)
r_zero = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {"role": "system", "content": "Je bent een klinisch chemicus."},
        {"role": "user", "content": same_case}
    ]
)

# Few-shot (met voorbeelden uit cel 4)
r_few = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {"role": "system", "content": FEW_SHOT_PROMPT},
        {"role": "user", "content": same_case}
    ]
)

print("=== ZERO-SHOT ===")
print(r_zero['message']['content'])
print("\n=== FEW-SHOT ===")
print(r_few['message']['content'])

=== ZERO-SHOT ===
Op basis van de gegeven biologische waardeheden en het gebruik van diuretica bij de patiënt, kunnen we een aantal opmerkingen maken:

1. Kalium (2.9 mmol/L):
   - Deze waarde is lager dan de normale bereik (3.5-5.0 mmol/L).
   - Het kan aangeven dat de diuretica invloed hebben op kalium-waarden.
   - Er is een risicovolle situatie mogelijk, aangezien lage kaliumwaarden (hypokalemië) gevaarlijk kunnen zijn.

2. Natrium (141 mmol/L):
   - Dit is binnen de normale bereikswaarden voor natrium (135-145 mmol/L).
   - De huidige waarde leek stabiel en ziet eruit als een goed gehalte.

3. Creatinine (88 µmol/L):
   - Deze waarde is lager dan de normale bereikswaarden voor creatinine (80-175 µmol/L in het algemeen).
   - Het kan aangeven dat de nierfunctie mogelijk beter is dan gemiddeld, of dat er een correctiefactor nodig is.

Overwegingen:
- Diuretica kunnen invloed hebben op kaliumwaarden. Er zou worden gecontroleerd op hypokalemië.
- Creatininereductie kan aangeven dat ni

In [11]:
# Cel 6: Resultaten opslaan voor later gebruik in evaluatie
import json
from datetime import datetime

results = {
    "timestamp": datetime.now().isoformat(),
    "model": "qwen2.5:7b",
    "experiments": [
        {
            "type": "zero_shot",
            "input": same_case,
            "output": r_zero['message']['content']
        },
        {
            "type": "few_shot", 
            "input": same_case,
            "output": r_few['message']['content']
        }
    ]
}

with open("../data/synthetic/experiment_01_results.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Opgeslagen!")

Opgeslagen!


In [14]:
pwd

'/Users/benjaminverbaan/Desktop/RAG/labconsult-ai/labconsult-ai/notebooks'

In [17]:
!git add Desktop/RAG/labconsult-ai/labconsult-ai/notebooks/01_basis_llm_calls.ipynbdata/synthetic/experiment_01_results.json





fatal: pathspec 'notebooks/01_basis_llm_calls.ipynb' did not match any files


In [18]:
pwd

'/Users/benjaminverbaan/Desktop/RAG/labconsult-ai/labconsult-ai/notebooks'

In [19]:
!git rev-parse --show-toplevel

/Users/benjaminverbaan/Desktop/RAG/labconsult-ai/labconsult-ai
